In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.distributions.laplace import Laplace
from torch.distributions.exponential import Exponential

# 1. Setup Model & Data
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Load a sufficiently long block of text
text = """Differential Privacy (DP) has emerged as the gold standard for privacy-preserving data analysis, providing a rigorous mathematical framework to quantify and limit the disclosure of individual information in statistical databases. At its core, DP addresses the fundamental tension between the utility of data—our ability to derive meaningful insights and societal benefits—and the privacy of the individuals whose data constitutes those datasets. Unlike previous "ad-hoc" methods such as k-anonymity or simple de-identification, which have repeatedly been defeated by linkage attacks and auxiliary information, differential privacy offers a provable guarantee that remains robust regardless of what an adversary might know.



### The Mathematical Foundation



The formal definition of differential privacy centers on the concept of "neighboring databases." Two databases, D and D', are considered neighbors if they differ by exactly one record (i.e., one person's data). A randomized algorithm M satisfies (epsilon, delta)-differential privacy if, for all neighboring databases D, D' and all possible subsets of outputs S:



P(M(D) ∈ S) ≤ exp(epsilon) * P(M(D') ∈ S) + delta



The parameter epsilon represents the "privacy budget." A smaller epsilon indicates a tighter privacy guarantee, meaning the distributions of outputs for D and D' are nearly indistinguishable. The parameter delta accounts for a small probability that the privacy guarantee might fail, typically set to be much smaller than 1/|D|.



### Mechanisms and Noise Injection



To achieve DP, one must inject controlled noise into the computation. The amount of noise required is determined by the Global Sensitivity of the function f, denoted as Δf. This measures the maximum amount the output of f can change by adding or removing a single entry.



Common mechanisms for achieving DP include:

* The Laplace Mechanism: Suitable for numeric queries, it adds noise sampled from a Laplace distribution centered at zero with a scale of Δf / epsilon.

* The Gaussian Mechanism: Often used in iterative algorithms like Deep Learning, it adds Gaussian noise. While it requires the delta parameter, it often provides better "composition" properties when performing many queries.

* The Exponential Mechanism: Used when the output is non-numeric (e.g., picking the most popular color in a dataset). It selects an output r with probability proportional to exp(epsilon * u(D, r) / 2Δu), where u is a utility function.



### Advanced Composition and zCDP



In complex workflows, such as training a neural network, we perform thousands of private operations. Standard composition (adding up the epsilons) is often too conservative, leading to a massive "privacy loss" that destroys utility. To combat this, researchers use Rényi Differential Privacy (RDP) or zero-Concentrated Differential Privacy (zCDP).



These frameworks view privacy loss as a random variable. By analyzing the moments of this variable, we can achieve much tighter bounds on the total privacy leakage over time. This is particularly crucial for DP-SGD (Stochastic Gradient Descent), where noise is added to the gradients during every training step.



### Privacy-Utility Trade-offs in Practice



Implementing DP is not without cost. The added noise introduces a "utility gap." In some cases, such as very small datasets or high-dimensional queries, the noise may overwhelm the signal. However, modern techniques like the Matrix Mechanism seek to optimize the noise distribution by identifying "workloads" of queries, effectively reducing error by correlating the noise across related questions.



Furthermore, differential privacy is being integrated into modern optimization paradigms. For instance, in Direct Preference Optimization (DPO)—a technique used to align large language models—researchers are exploring DP-DPO to ensure that the fine-tuning process does not leak the sensitive preferences or identities of the human annotators involved in the training loop.



### Societal and Ethical Implications



The transition from "privacy by policy" to "privacy by math" represents a paradigm shift. Major institutions, including the U.S. Census Bureau and tech giants like Google and Apple, have deployed DP to protect user telemetry and demographic data. 



However, DP also forces a transparent conversation about trade-offs. By choosing an epsilon, a policy maker is explicitly deciding how much "risk" is acceptable for a specific "gain" in accuracy. This transparency is often uncomfortable, as it quantifies exactly how much we are willing to "sacrifice" privacy for the sake of data-driven decision-making. 



### Future Horizons



As we move toward a world dominated by massive AI models, the intersection of DP and Generative AI becomes critical. Can we generate synthetic data that maintains the statistical properties of a real dataset without ever exposing an individual's record? Can we ensure that a Large Language Model (LLM) doesn't "memorize" a user's private email address during training?



Differential privacy provides the tools to answer these questions. By treating privacy as a bounded, consumable resource, it allows us to build systems that are not only intelligent but also fundamentally respectful of individual autonomy. The challenge of the next decade lies in refining these mechanisms to minimize the utility cost, making privacy-preserving computation the default standard for all data science."""
inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(device)

# 2. Get Clean Logits
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, :-1, :]

seq_len, vocab_size = logits.shape

# 3. Calculate True Ranks (Rank 1 is the argmax)
sorted_indices = torch.argsort(logits, dim=-1, descending=True)
ranks = torch.zeros_like(sorted_indices)
ranks.scatter_(dim=-1, index=sorted_indices, src=torch.arange(vocab_size, device=device).expand(seq_len, -1))

# --- Baseline Non-Private ---
temperature = 1.0
probs = F.softmax(logits / temperature, dim=-1)
baseline_selections = torch.multinomial(probs, num_samples=1).squeeze(-1)
baseline_ranks = ranks[torch.arange(seq_len), baseline_selections].cpu().numpy() + 1
sorted_baseline = np.sort(baseline_ranks)

# 4. Define Noise Generation Functions
def get_gumbel_noise(shape, scale, device):
    u = torch.rand(shape, device=device)
    return -scale * torch.log(-torch.log(u))

def get_laplace_noise(shape, scale, device):
    m = Laplace(torch.tensor([0.0]).to(device), torch.tensor([scale]).to(device))
    return m.sample(shape).squeeze(-1)

def get_exponential_noise(shape, scale, device):
    m = Exponential(torch.tensor([1.0/scale]).to(device))
    return m.sample(shape).squeeze(-1)

def get_normal_noise(shape, scale, device):
    return torch.randn(shape, device=device) * scale

# 5. Experimental Parameters
d = 10.0  # Sensitivity (Delta f)
delta = 1e-5 # Delta for (epsilon, delta)-DP (Gaussian Mechanism)
epsilons = [1.0, 5.0, 10.0]

noise_mechanisms = {
    "Gumbel_ExpMech": get_gumbel_noise,
    "Laplace": get_laplace_noise,
    "RawExponential": get_exponential_noise,
    "Gaussian": get_normal_noise
}

# Apply d-clip once
max_logits = logits.max(dim=-1, keepdim=True)[0]
clipped_logits = max_logits - torch.clamp(max_logits - logits, max=d)

# Base plotting setup
sns.set_theme(style="whitegrid")
y_vals = np.arange(1, seq_len + 1) / seq_len
palette = sns.color_palette("rocket", len(epsilons))

# 6. Run Experiments and Generate Plots per Distribution
for dist_name, noise_func in noise_mechanisms.items():
    print(f"Processing {dist_name} distribution...")
    
    dist_results = {}
    
    for eps in epsilons:
        # Calibrate the scale based on the mechanism
        if dist_name == "Gaussian":
            # standard Gaussian mechanism bound: sigma = (Delta_f / epsilon) * sqrt(2 * ln(1.25 / delta))
            multiplier = math.sqrt(2 * math.log(1.25 / delta))
            scale = (d / eps) * multiplier
        else:
            # Pure epsilon-DP scale
            scale = d / eps
            
        noise = noise_func(clipped_logits.shape, scale, device)
        noisy_logits = clipped_logits + noise
        selections = torch.argmax(noisy_logits, dim=-1)
        dist_results[eps] = ranks[torch.arange(seq_len), selections].cpu().numpy() + 1
        
    # --- Create Histogram Plot ---
    fig_hist, ax_hist = plt.subplots(figsize=(10, 6))
    
    # Plot Baseline
    sns.histplot(baseline_ranks, log_scale=True, color="black", alpha=0.3, label="Baseline", ax=ax_hist, element="step", fill=True)
    
    # Plot Noisy Distributions
    for i, eps in enumerate(epsilons):
        sns.histplot(dist_results[eps], log_scale=True, color=palette[i], alpha=0.4, label=f"$\epsilon$ = {eps}", ax=ax_hist, element="step", fill=False, linewidth=2)
        
    ax_hist.set_xlabel("Token Rank (Log Scale)")
    ax_hist.set_ylabel("Count")
    ax_hist.set_title(f"Rank Histogram: {dist_name} Mechanism (d-clip={d})")
    ax_hist.legend()
    
    hist_filename = f"hist_{dist_name}.png"
    plt.tight_layout()
    fig_hist.savefig(hist_filename, dpi=300)
    plt.close(fig_hist)
    
    # --- Create CDF Plot ---
    fig_cdf, ax_cdf = plt.subplots(figsize=(10, 6))
    
    # Plot Baseline
    ax_cdf.plot(sorted_baseline, y_vals, color="black", linewidth=2, linestyle="--", label="Baseline")
    
    # Plot Noisy Distributions
    for i, eps in enumerate(epsilons):
        sorted_ranks = np.sort(dist_results[eps])
        ax_cdf.plot(sorted_ranks, y_vals, color=palette[i], linewidth=2.5, label=f"$\epsilon$ = {eps}")
        
    ax_cdf.set_xscale("log")
    ax_cdf.set_xlabel("Token Rank (Log Scale)")
    ax_cdf.set_ylabel("Cumulative Probability")
    ax_cdf.set_title(f"CDF Suboptimality: {dist_name} Mechanism")
    ax_cdf.legend()
    
    cdf_filename = f"cdf_{dist_name}.png"
    plt.tight_layout()
    fig_cdf.savefig(cdf_filename, dpi=300)
    plt.close(fig_cdf)

print("✅ All 8 plots have been successfully generated and saved.")

<>:188: SyntaxWarning: invalid escape sequence '\e'
<>:209: SyntaxWarning: invalid escape sequence '\e'
<>:188: SyntaxWarning: invalid escape sequence '\e'
<>:209: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_23/2510615724.py:188: SyntaxWarning: invalid escape sequence '\e'
  sns.histplot(dist_results[eps], log_scale=True, color=palette[i], alpha=0.4, label=f"$\epsilon$ = {eps}", ax=ax_hist, element="step", fill=False, linewidth=2)
/tmp/ipykernel_23/2510615724.py:209: SyntaxWarning: invalid escape sequence '\e'
  ax_cdf.plot(sorted_ranks, y_vals, color=palette[i], linewidth=2.5, label=f"$\epsilon$ = {eps}")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Processing Gumbel_ExpMech distribution...
Processing Laplace distribution...
Processing RawExponential distribution...
Processing Gaussian distribution...
✅ All 8 plots have been successfully generated and saved.
